In [ ]:
%pip install trl transformers huggingface_hub
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch

from huggingface_hub import login
# start interactive login (or use login(token="YOUR_TOKEN") if you prefer)
login(token="")
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load dataset
dataset = load_dataset("HuggingFaceTB/smoltalk", "all")

# Configure model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name).to(
    device
)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Set padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

# Set up chat template if not present
if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% for message in messages %}{{message['role'] + ': ' + message['content'] + '\n'}}{% endfor %}"

# TEST BEFORE TRAINING
print("="*50)
print("TESTING BEFORE TRAINING")
print("="*50)
prompt = "Write a haiku about programming"
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("="*50)

# Configure trainer
training_args = SFTConfig(
    output_dir="./sft_output",
    max_steps=100,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].select(range(1000)),
    eval_dataset=dataset["test"].select(range(1000)),
    processing_class=tokenizer,
)

# Start training
trainer.train()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


TESTING BEFORE TRAINING
Before training:
user: Write a haiku about programming

Write a haiku about programming

The haiku is a Japanese poem that is written in 5-7-5 syllables. It is a very simple form of poetry that is easy to write and read. It is a very simple form of poetry that is easy to write and read. It is a very simple form of poetry that is easy to write and read. It is a very simple form of poetry that is easy to write and read. It is a very simple form of poetry


Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (9435 > 8192). Running this sequence through the model will result in indexing errors


Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss
50,1.648527,1.726740
100,1.782274,1.722056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=1.708188304901123, metrics={'train_runtime': 1011.6403, 'train_samples_per_second': 0.791, 'train_steps_per_second': 0.099, 'total_flos': 521037553646592.0, 'train_loss': 1.708188304901123})

In [ ]:
print("Checking if colab works or not")
if torch.cuda.is_available():
    device = "cuda" 
    print("cuda gpu is available")
else:
    device = "cpu"

Checking if colab works or not
cuda gpu is available
